In [1]:
%pwd

'd:\\Siam\\KIdney disease\\research'

In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\KIdney disease'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class XAIConfig:
    root_dir: Path
    dataset_dir: Path
    model_path: Path
    output_dir: Path
    num_samples: int

In [6]:
from kidney_disease_classification.constants import *
from kidney_disease_classification.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config["artifacts_root"]])

    def get_xai_config(self) -> XAIConfig:
        config = self.config["xai"]

        create_directories([config["root_dir"], config["output_dir"]])

        evaluation_config = XAIConfig(
            root_dir=config["root_dir"],
            dataset_dir=config["dataset_dir"],
            model_path=config["model_path"],
            output_dir=config["output_dir"],
            num_samples=config["num_samples"]
        )
        return evaluation_config

In [8]:
import os
import sys
import random
import torch
import torch.nn as nn
import numpy as np
import cv2

from pathlib import Path
from PIL import Image
from torchvision import transforms, models

from kidney_disease_classification.exception import CustomException
from kidney_disease_classification.logger import logging

In [9]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate_cam(self, input_tensor, class_idx=None):
        self.model.zero_grad()

        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = 0

        score = output[:, class_idx]
        score.backward(retain_graph=True)

        gradients = self.gradients
        activations = self.activations

        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1).squeeze()

        cam = torch.relu(cam)

        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam

In [10]:
class XAI:
    def __init__(self, config: XAIConfig, params):
        self.config = config
        self.params = params

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------------------------
    # 1. LOAD MODEL
    # ---------------------------
    def load_model(self):
        try:
            model = models.efficientnet_b0(pretrained=False)

            in_features = model.classifier[1].in_features
            model.classifier[1] = nn.Linear(in_features, self.params["NUM_CLASSES"])

            model.load_state_dict(torch.load(self.config.model_path, map_location=self.device))
            model.to(self.device)
            model.eval()

            logging.info("Model loaded successfully for XAI.")
            return model

        except Exception as e:
            raise CustomException(e, sys)

    # ---------------------------
    # 2. GET TRANSFORM
    # ---------------------------
    def get_transform(self):
        img_size = tuple(self.params["IMAGE_SIZE"])

        transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

        return transform

    # ---------------------------
    # 3. LOAD RANDOM TEST IMAGES
    # ---------------------------
    def get_sample_images(self):
        test_dir = Path(self.config.dataset_dir) / "test"

        all_images = list(test_dir.rglob("*.jpg")) + list(test_dir.rglob("*.png")) + list(test_dir.rglob("*.jpeg"))

        if len(all_images) == 0:
            raise CustomException(f"No images found in {test_dir}", sys)

        random.shuffle(all_images)

        num_samples = min(self.config.num_samples, len(all_images))
        return all_images[:num_samples]

    # ---------------------------
    # 4. OVERLAY HEATMAP
    # ---------------------------
    def overlay_heatmap(self, original_img, cam):
        original_img = np.array(original_img)

        if len(original_img.shape) == 2:
            original_img = cv2.cvtColor(original_img, cv2.COLOR_GRAY2RGB)

        cam_resized = cv2.resize(cam, (original_img.shape[1], original_img.shape[0]))

        heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

        overlay = cv2.addWeighted(original_img, 0.6, heatmap, 0.4, 0)

        return overlay

    # ---------------------------
    # 5. MAIN FUNCTION
    # ---------------------------
    def generate_gradcam_outputs(self):
        try:
            logging.info("Starting Grad-CAM XAI stage...")

            os.makedirs(self.config.output_dir, exist_ok=True)

            model = self.load_model()
            transform = self.get_transform()

            # EfficientNet target conv layer
            target_layer = model.features[-1]

            gradcam = GradCAM(model, target_layer)

            sample_images = self.get_sample_images()

            for img_path in sample_images:
                img = Image.open(img_path).convert("RGB")
                input_tensor = transform(img).unsqueeze(0).to(self.device)

                output = model(input_tensor).squeeze()
                prob = torch.sigmoid(output).item()

                pred_label = "disease" if prob > 0.5 else "normal"

                cam = gradcam.generate_cam(input_tensor)

                overlay = self.overlay_heatmap(img, cam)

                save_path = os.path.join(
                    self.config.output_dir,
                    f"{img_path.stem}_pred_{pred_label}_prob_{prob:.2f}.png"
                )

                Image.fromarray(overlay).save(save_path)

                logging.info(f"Saved Grad-CAM: {save_path}")

            logging.info("Grad-CAM XAI stage completed successfully.")
            return True

        except Exception as e:
            raise CustomException(e, sys)

In [11]:
try:
    config = ConfigurationManager()
    xai_config = config.get_xai_config()
    xai = XAI(config=xai_config, params=config.params)
    xai.generate_gradcam_outputs()
except Exception as e:
    raise CustomException(e, sys)

[2026-05-14 15:22:26,344: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-14 15:22:26,347: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-14 15:22:26,349: INFO: common: created directory at: artifacts]
[2026-05-14 15:22:26,350: INFO: common: created directory at: artifacts/xai]
[2026-05-14 15:22:26,352: INFO: common: created directory at: artifacts/xai/gradcam_outputs]
[2026-05-14 15:22:26,352: INFO: 222487041: Starting Grad-CAM XAI stage...]


d:\Siam\KIdney disease\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Siam\KIdney disease\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


[2026-05-14 15:22:26,694: INFO: 222487041: Model loaded successfully for XAI.]


d:\Siam\KIdney disease\env\Lib\site-packages\torch\nn\modules\module.py:1870: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


[2026-05-14 15:22:27,248: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\Normal- (690)_pred_normal_prob_0.42.png]
[2026-05-14 15:22:27,489: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\Normal- (804)_pred_normal_prob_0.49.png]
[2026-05-14 15:22:27,797: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\Normal- (684)_pred_normal_prob_0.50.png]
[2026-05-14 15:22:28,045: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\Tumor- (732)_pred_disease_prob_0.57.png]
[2026-05-14 15:22:28,297: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\Tumor- (798)_pred_disease_prob_0.52.png]
[2026-05-14 15:22:28,523: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\Normal- (749)_pred_normal_prob_0.45.png]
[2026-05-14 15:22:28,737: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\Normal- (776)_pred_normal_prob_0.47.png]
[2026-05-14 15:22:28,969: INFO: 222487041: Saved Grad-CAM: artifacts/xai/gradcam_outputs\T